In [2]:
print("hi this was really intense")

hi this was really intense


In [1]:
%pwd

'c:\\Users\\PC143\\Desktop\\local folder\\Medical-chatbot\\research'

In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'c:\\Users\\PC143\\Desktop\\local folder\\Medical-chatbot'

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\PC143\AppData\Local\Temp\ipykernel_14328\2072117745.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, PyMuPDFLoader


In [4]:
def load_pdf_files(Data):
    loader = DirectoryLoader(
        Data,
        glob="*.pdf",
        loader_cls=PyMuPDFLoader
    )
    Documents = loader.load()
    return Documents


In [5]:
extracted_data = load_pdf_files("Data")

In [6]:
len(extracted_data)

4505

In [7]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of document objects, return a new list of document objects containg
    only "source" in the metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    
    for doc in docs:
        src = doc.metadata.get("source")
        
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
        
    return minimal_docs

#this is a filter operation to make the data clean

In [8]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [9]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    texts_chunks = text_splitter.split_documents(minimal_docs)
    return texts_chunks

In [10]:
text_chunks = text_split(minimal_docs)
len(text_chunks)

40135

In [14]:
import os
import certifi

os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings


In [16]:
embeddings = download_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6440.65it/s]


In [17]:
vector = embeddings.embed_query("hellow world")
# testing
len(vector)

384

In [18]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [19]:
PINECONE_API_KEY= os.getenv("PINECONE_API_KEY")
GROQ_API_KEY= os.getenv("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [20]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key = pinecone_api_key)
pc

In [25]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name= index_name,
        dimension= 384,  #dimension of embeddings
        metric= "cosine",
        spec= ServerlessSpec(cloud="aws", region="us-east-1")
    )

Index = pc.Index(index_name)    


In [26]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [27]:
#load exiting index

from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name=index_name
)

To add more documents in the existing_index

In [28]:
extra = Document(
    page_content="so this is how we add more data into our existing index",
    metadata={"source": "vscode"}
)

In [29]:
docsearch.add_documents(documents=[extra])

['f1432366-1d36-4c62-8c71-d1c84a280255']

In [32]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='02177ca8-53fb-4ff0-acd7-cbe248e5cb5e', metadata={'source': 'Data\\encyclopedia-of-medicine-vol-1-5-3rd-edition.pdf'}, page_content='forms of acne.\nPurpose\nDifferent types of antiacne drugs are used for\ndifferent purposes. For example, lotions, soaps, gels,\nand creams containing benzoyl peroxide or tretinoin\nmay be used to clear up mild to moderately severe\nacne. Isotretinoin (Accutane) is prescribed only for\nvery severe, disfiguring acne.\nAcne is a skin condition that occurs when pores or\nhair follicles become blocked. This blockage allows a\nwaxy material called sebum to collect inside the pores'),
 Document(id='b534bcfb-5e5e-4f5f-b793-8ff19ca55fae', metadata={'source': 'Data\\encyclopedia-of-medicine-vol-1-5-3rd-edition.pdf'}, page_content='Acne folliculitis. (Custom Medical Stock Photo. Reproduced by\npermission.)\nG A L E E N C Y C L O P E D I A O F M E D I C I N E\n1503\nFood allergies'),
 Document(id='a1c796ab-ad1a-4535-a030-f46a97a73b53', metadata={'source

In [37]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY
)

In [35]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [38]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response_text = rag_chain.invoke("what is Acromegaly and gigantism?")
print(response_text)

Acromegaly is a disorder caused by excess release of growth hormone from the pituitary gland, leading to abnormal enlargement of bone and soft tissue and various systemic effects. When this hormone excess occurs before the growth plates close in children, it produces gigantism, resulting in extreme height. Both conditions reflect hyperpituitarism (over‑activity of the pituitary).
